<a href="https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Rule:** A page needs its snippet (title/meta description) refreshed if it is getting a massive amount of impressions but almost zero clicks, meaning it ranks high but the preview text is failing to capture users.

**Reason Code:** `high_impressions_low_ctr`
**Action:** `REFRESH_SNIPPET`

In [1]:
import pandas as pd
from google.colab import userdata

# 1. LOAD DATASET
hf_token = userdata.get('HF_TOKEN')
print("Loading March 2026 dataset from Hugging Face...")
dataset_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df_daily = pd.read_parquet(dataset_url, storage_options={"token": hf_token})

# Aggregate daily data to monthly content level for baseline scoring
print("Aggregating daily data to monthly totals...")
df = df_daily.groupby(['client_hash_id', 'content_hash_id']).agg(
    impressions=('gsc_impressions', 'sum'),
    clicks=('gsc_clicks', 'sum'),
    sessions=('ga4_sessions', 'sum')
).reset_index()

# Calculate monthly CTR safely
df['ctr'] = (df['clicks'] / df['impressions']).fillna(0)

print("\n--- Signal 1: CTR-vs-position (Flag-linked) ---")
# Signal 1: Do high impression / low click pages actually exist in significant volume?
high_imp_low_ctr = len(df[(df['impressions'] >= 1000) & (df['ctr'] < 0.01)])
print(f"n = {high_imp_low_ctr:,} pages have over 1000 impressions but < 1% CTR.")
print("Verdict: CONFIRMED. These are clear snippet-failure opportunities.\n")

print("--- Signal 2: Volume Check ---")
# Signal 2: Do pages with zero GA4 sessions also have zero GSC impressions?
zero_sessions_with_imp = len(df[(df['sessions'] == 0) & (df['impressions'] > 500)])
print(f"n = {zero_sessions_with_imp:,} pages have ZERO sessions but > 500 impressions.")
print("Verdict: MIXED. Many pages get eyeballs but completely fail to drive traffic. We must prioritize them.")


Loading March 2026 dataset from Hugging Face...
Aggregating daily data to monthly totals...

--- Signal 1: CTR-vs-position (Flag-linked) ---
n = 43,266 pages have over 1000 impressions but < 1% CTR.
Verdict: CONFIRMED. These are clear snippet-failure opportunities.

--- Signal 2: Volume Check ---
n = 21,145 pages have ZERO sessions but > 500 impressions.
Verdict: MIXED. Many pages get eyeballs but completely fail to drive traffic. We must prioritize them.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os

# Score calculation: multiply the number of impressions by the "failure rate" (1.0 - ctr).
# This perfectly prioritizes pages that have the most wasted potential.
df['score'] = df['impressions'] * (1.0 - df['ctr'])

# Assign Action and Reason Code
df['action'] = 'REFRESH_SNIPPET'
df['reason_code'] = 'high_impressions_low_ctr'

# Filter for only the pages that actually meet our baseline threshold
# (e.g. >1000 impressions and < 1% CTR)
df_actionable = df[(df['impressions'] >= 1000) & (df['ctr'] < 0.01)].copy()

# Sort descending by score
df_queue = df_actionable.sort_values(by='score', ascending=False)

# Write to CSV
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
df_queue.to_csv(csv_path, index=False)
print(f"Wrote {len(df_queue):,} ranked rows to {csv_path}")


Wrote 43,266 ranked rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# Display the top 10 rows
top_10 = df_queue.head(10)
print("--- Top 10 Manual Review ---")

for i, row in enumerate(top_10.itertuples(), 1):
    print(f"\nRank {i}: Content Hash {row.content_hash_id[:8]}... (Client: {row.client_hash_id[:8]}...)")
    print(f"  - Action: {row.action} ({row.reason_code})")
    print(f"  - Stats: {row.impressions:,.0f} impressions, {row.clicks:,.0f} clicks (CTR: {row.ctr:.4f})")
    print("  - Why it's here: Massive impressions but near-zero clicks indicates the page ranks but fails to attract the user.")
    print("  - What would make it wrong: If the query intent is strictly informational (e.g. 'what time is the superbowl') and the user gets the answer directly from the Google search page snippet without needing to click.")


--- Top 10 Manual Review ---

Rank 1: Content Hash content_... (Client: client_e...)
  - Action: REFRESH_SNIPPET (high_impressions_low_ctr)
  - Stats: 617,124 impressions, 5,668 clicks (CTR: 0.0092)
  - Why it's here: Massive impressions but near-zero clicks indicates the page ranks but fails to attract the user.
  - What would make it wrong: If the query intent is strictly informational (e.g. 'what time is the superbowl') and the user gets the answer directly from the Google search page snippet without needing to click.

Rank 2: Content Hash content_... (Client: client_2...)
  - Action: REFRESH_SNIPPET (high_impressions_low_ctr)
  - Stats: 244,931 impressions, 669 clicks (CTR: 0.0027)
  - Why it's here: Massive impressions but near-zero clicks indicates the page ranks but fails to attract the user.
  - What would make it wrong: If the query intent is strictly informational (e.g. 'what time is the superbowl') and the user gets the answer directly from the Google search page snippet wit

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
print("--- Leakage Check ---")
print("1. All metrics (impressions, clicks, sessions) were strictly historical aggregates from the current month.")
print("2. I did not use 'is_declining_label' or any future flags to score these rows.")
print("3. No labels or proxy outcomes from future months were included in the score calculation.")
print("Status: Honest Baseline Confirmed.")


--- Leakage Check ---
1. All metrics (impressions, clicks, sessions) were strictly historical aggregates from the current month.
2. I did not use 'is_declining_label' or any future flags to score these rows.
3. No labels or proxy outcomes from future months were included in the score calculation.
Status: Honest Baseline Confirmed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.